In [0]:
%pip install nltk spacy pandas scikit-learn datasets 'fsspec>=2024.2.0' 'huggingface-hub>=0.28.0' --upgrade

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import nltk
import subprocess

# Download spacy model
subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)

# Download NLTK data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

print("All downloads completed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 186.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/spark-7479725a-789b-4283-9bce-6e/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     /home/spark-7479725a-789b-4283-9bce-6e/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/spark-7479725a-789b-4283-9bce-6e/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


All downloads completed successfully!


In [0]:
import pandas as pd
from datasets import load_dataset

# a. Load the IMDb Movie Reviews dataset (train split)
dataset = load_dataset('stanfordnlp/imdb', split='train')

# b. Create a dataframe with specific columns
# We convert the Hugging Face dataset to a Pandas DataFrame first
df = pd.DataFrame(dataset)
df = df.rename(columns={'text': 'review_text', 'label': 'sentiment'})

# c. Map sentiment labels (0 -> 'negative', 1 -> 'positive')
sentiment_map = {0: 'negative', 1: 'positive'}
df['sentiment'] = df['sentiment'].map(sentiment_map)

# d. Save the processed dataset as movie_reviews.csv
# index=False prevents pandas from adding a column for the row numbers
df.to_csv('movie_reviews.csv', index=False)

print("File 'movie_reviews.csv' has been created successfully!")
print(f"Dataset shape: {df.shape}")
print("\nFirst few rows:")
display(df.head())

/databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:56: UserWarning: The cache_dir for this dataset is /tmp/.hf.data.cache, which is not a persistent path.Therefore, if/when the cluster restarts, the downloaded dataset will be lost.The persistent storage options for this workspace/cluster config are: [UC Volumes].Please update either `cache_dir` or the environment variable `HF_DATASETS_CACHE`to be under one of the following root directories: ['/Volumes/']
  warnings.warn(warning_message)


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

/databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:24: UserWarning: During large dataset downloads, there could be multiple progress bar widgets that can cause performance issues for your notebook or browser. To avoid these issues, use `datasets.utils.logging.disable_progress_bar()` to turn off the progress bars.
  warnings.warn(


plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

File 'movie_reviews.csv' has been created successfully!
Dataset shape: (25000, 2)

First few rows:


,review_text,sentiment
0,I rented I AM CURIOUS-YELLOW from my video sto...,negative
1,"""I Am Curious: Yellow"" is a risible and preten...",negative
2,If only to avoid making this type of film in t...,negative
3,This film was probably inspired by Godard's Ma...,negative
4,"Oh, brother...after hearing about this ridicul...",negative


In [0]:
import string
from nltk.corpus import stopwords

# Load the dataset we just saved
df = pd.read_csv('movie_reviews.csv')

# Get English stop words
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """
    Preprocess text by:
    1. Converting to lowercase
    2. Removing punctuation
    3. Removing stop words
    """
    # a. Convert to lowercase
    text = text.lower()
    
    # b. Remove punctuation
    translator = str.maketrans('', '', string.punctuation)
    text = text.translate(translator)
    
    # c. Remove stop words
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    
    return ' '.join(filtered_words)

# Apply preprocessing to all reviews
df['processed_text'] = df['review_text'].apply(preprocess_text)

print("Text preprocessing completed!")
print("\nExample comparison:")
print("\nOriginal text:")
print(df['review_text'].iloc[0][:300])
print("\n" + "="*80)
print("\nProcessed text:")
print(df['processed_text'].iloc[0][:300])

Text preprocessing completed!

Example comparison:

Original text:
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h


Processed text:
rented curiousyellow video store controversy surrounded first released 1967 also heard first seized us customs ever tried enter country therefore fan films considered controversial really see myselfbr br plot centered around young swedish drama student named lena wants learn everything life particul


## Task 1: Text Preprocessing - Conceptual Questions

### 1. What is the purpose of text preprocessing in NLP applications?

Text preprocessing is essential for:
* **Normalization**: Reducing variations in text to create consistency (e.g., "Good" and "good" become the same)
* **Noise Reduction**: Removing irrelevant information (punctuation, special characters) that doesn't contribute to meaning
* **Dimensionality Reduction**: Decreasing the vocabulary size by removing stop words and standardizing text
* **Feature Extraction**: Preparing text for machine learning models by converting it to a format algorithms can process
* **Improved Model Performance**: Cleaner data leads to better pattern recognition and more accurate predictions

### 2. How does stop word removal affect text analysis?

**Positive Effects**:
* Reduces vocabulary size significantly (stop words like "the", "is", "and" are very common)
* Focuses analysis on content-bearing words that carry more meaning
* Speeds up processing and reduces memory requirements
* Can improve model performance by reducing noise

**Considerations**:
* Removes grammatical structure that might be important in some contexts
* May lose nuance in sentiment analysis (e.g., "not good" becomes "good")

### 3. Why is case normalization important in text preprocessing?

* **Consistency**: Treats "Hello", "hello", and "HELLO" as the same word
* **Vocabulary Reduction**: Prevents the same word from being counted multiple times due to different capitalization
* **Pattern Recognition**: Helps models identify patterns more easily without case variations
* **Fairness**: Ensures position in sentence (start vs middle) doesn't affect word treatment

### 4. What are potential drawbacks of removing stop words?

* **Loss of Context**: Phrases like "not bad" or "no good" lose their meaning
* **Sentiment Analysis Issues**: Negations ("not", "no") are crucial for understanding sentiment
* **Named Entity Problems**: "The Who" (band name) becomes just "Who"
* **Question Understanding**: Question words ("who", "what", "where") are often stop words
* **Grammar Loss**: Sentence structure information is completely removed
* **Over-aggressive Filtering**: May remove words that are important in specific domains

In [0]:
from nltk.tokenize import sent_tokenize, word_tokenize

# a. Perform sentence tokenization
df['tokenized_sentences'] = df['review_text'].apply(sent_tokenize)

# b. Perform word tokenization on original text
df['tokenized_words'] = df['review_text'].apply(word_tokenize)

# Also tokenize the processed text for later use
df['tokenized_words_processed'] = df['processed_text'].apply(lambda x: x.split())

print("Tokenization completed!")
print("\nExample from first review:")
print("\nOriginal text:")
print(df['review_text'].iloc[0][:200])
print("\n" + "="*80)
print(f"\nNumber of sentences: {len(df['tokenized_sentences'].iloc[0])}")
print(f"\nFirst 3 sentences:")
for i, sent in enumerate(df['tokenized_sentences'].iloc[0][:3], 1):
    print(f"{i}. {sent}")
print("\n" + "="*80)
print(f"\nNumber of words (original): {len(df['tokenized_words'].iloc[0])}")
print(f"\nFirst 20 words:")
print(df['tokenized_words'].iloc[0][:20])

Tokenization completed!

Example from first review:

Original text:
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev


Number of sentences: 9

First 3 sentences:
1. I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967.
2. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life.
3. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States.


Number o

## Task 2: Text Tokenization - Conceptual Questions

### 1. What is the difference between word and sentence tokenization?

**Word Tokenization**:
* Splits text into individual words or tokens
* Handles punctuation, contractions, and special characters
* Example: "Hello, world!" → ["Hello", ",", "world", "!"]
* Used for: word frequency analysis, vocabulary building, text classification

**Sentence Tokenization**:
* Splits text into complete sentences
* Handles sentence boundaries (periods, question marks, exclamation points)
* Challenges: abbreviations (Dr., Mr.), decimals (3.14), ellipses (...)
* Example: "Hello world. How are you?" → ["Hello world.", "How are you?"]
* Used for: document summarization, sentence-level analysis, machine translation

### 2. Why is tokenization important for text analysis?

* **Foundation for NLP**: First step in most text processing pipelines
* **Enables Counting**: Can't count words/sentences without identifying them first
* **Feature Extraction**: Tokens become features for machine learning models
* **Statistical Analysis**: Enables frequency analysis, n-gram generation, and pattern detection
* **Semantic Understanding**: Breaking text into meaningful units helps extract meaning
* **Language-Specific Processing**: Different languages tokenize differently (e.g., Chinese has no spaces)

### 3. How do regular expressions help in tokenization?

* **Pattern Matching**: Define rules for what constitutes a token (e.g., word boundaries)
* **Custom Rules**: Handle special cases like email addresses, URLs, hashtags
* **Flexibility**: Adapt to domain-specific text (social media, scientific papers, code)
* **Efficiency**: Fast pattern-based splitting without complex logic
* **Edge Cases**: Handle contractions ("don't" → "do" + "n't"), hyphenated words, possessives

**Example patterns**:
* `\w+`: Matches word characters
* `\s+`: Matches whitespace for splitting
* `[A-Za-z]+`: Matches only alphabetic characters

### 4. What challenges might you encounter when tokenizing social media text?

* **Informal Language**: Misspellings, abbreviations ("u" for "you", "brb" for "be right back")
* **Emoticons & Emojis**: 😊, :), :-D need special handling as meaningful units
* **Hashtags**: #MachineLearning should stay together, not split into words
* **Mentions**: @username should be treated as single token
* **URLs**: http://example.com should remain intact
* **Non-standard Punctuation**: Multiple punctuation ("!!!", "???"), no spaces between words
* **Slang & Neologisms**: New words and phrases that aren't in dictionaries
* **Code-Switching**: Mixing languages within same text
* **Character Repetition**: "soooooo goooood" for emphasis

In [0]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

# a. Create bag-of-words representation using CountVectorizer
vectorizer = CountVectorizer(max_features=1000)
bow_matrix = vectorizer.fit_transform(df['processed_text'])

print(f"Bag-of-Words matrix shape: {bow_matrix.shape}")
print(f"(Number of documents: {bow_matrix.shape[0]}, Number of features: {bow_matrix.shape[1]})")

# b. Calculate word frequencies
word_frequencies = np.asarray(bow_matrix.sum(axis=0)).flatten()

# c. Get vocabulary and create frequency pairs
vocabulary = vectorizer.get_feature_names_out()
word_freq_pairs = list(zip(vocabulary, word_frequencies))

# Sort by frequency (descending)
word_freq_pairs_sorted = sorted(word_freq_pairs, key=lambda x: x[1], reverse=True)

# Display top 20 most frequent words
print("\nTop 20 Most Frequent Words:")
print("="*50)
for i, (word, freq) in enumerate(word_freq_pairs_sorted[:20], 1):
    print(f"{i:2d}. {word:15s} : {freq:,} occurrences")

# Store word frequencies in a dataframe for later saving
word_freq_df = pd.DataFrame(word_freq_pairs_sorted, columns=['word', 'frequency'])

print("\nVectorization completed!")

Bag-of-Words matrix shape: (25000, 1000)
(Number of documents: 25000, Number of features: 1000)

Top 20 Most Frequent Words:
 1. br              : 57,145 occurrences
 2. movie           : 41,813 occurrences
 3. film            : 37,461 occurrences
 4. one             : 25,511 occurrences
 5. like            : 19,645 occurrences
 6. good            : 14,555 occurrences
 7. even            : 12,506 occurrences
 8. would           : 12,135 occurrences
 9. time            : 11,782 occurrences
10. really          : 11,663 occurrences
11. story           : 11,455 occurrences
12. see             : 11,223 occurrences
13. much            : 9,586 occurrences
14. well            : 9,378 occurrences
15. get             : 9,212 occurrences
16. also            : 9,073 occurrences
17. people          : 8,952 occurrences
18. bad             : 8,914 occurrences
19. great           : 8,894 occurrences
20. first           : 8,859 occurrences

Vectorization completed!


## Task 3: Basic Text Vectorization - Conceptual Questions

### 1. What information does a bag-of-words model capture?

**Captures**:
* **Word Presence**: Which words appear in a document
* **Word Frequency**: How many times each word appears
* **Vocabulary**: The complete set of unique words across all documents
* **Document Length**: Implicitly through total word counts
* **Topic Indicators**: Frequent words often indicate document topics

**Does NOT Capture**:
* **Word Order**: "dog bites man" and "man bites dog" are identical
* **Grammar**: No syntactic structure preserved
* **Semantics**: "good" and "excellent" are treated as completely different
* **Context**: "bank" (river) vs "bank" (financial) cannot be distinguished
* **Relationships**: Cannot capture phrases like "New York" as a single concept

### 2. How does vectorization help in text analysis?

* **Machine Learning Compatibility**: Converts text into numerical format that algorithms require
* **Mathematical Operations**: Enables distance calculations, similarity measurements, clustering
* **Standardization**: Creates fixed-length representations regardless of document length
* **Feature Engineering**: Each word becomes a feature for predictive modeling
* **Scalability**: Sparse matrices handle large vocabularies efficiently
* **Comparison**: Makes it possible to compare documents quantitatively
* **Dimensionality Management**: Controls feature space size (e.g., top 1000 words)

**Common Use Cases**:
* Text classification (spam detection, sentiment analysis)
* Document similarity and clustering
* Topic modeling
* Information retrieval

### 3. What are the limitations of basic frequency counts?

* **No Semantic Understanding**: Synonyms counted separately ("happy" ≠ "joyful")
* **Common Words Dominate**: Frequent but uninformative words get high weights
* **No Context**: Polysemy (multiple meanings) not handled
* **Ignores Importance**: A rare technical term may be more important than common word
* **Vocabulary Size**: Can become very large (millions of unique words)
* **Sparsity**: Most documents use small fraction of total vocabulary
* **No Word Order**: "not good" vs "good" both just count "not" and "good"
* **No Negation**: "I do not like this" and "I like this" look similar

**Better Alternatives**:
* **TF-IDF**: Weights words by inverse document frequency
* **Word Embeddings**: Word2Vec, GloVe capture semantic relationships
* **Transformers**: BERT, GPT capture context and meaning

In [0]:
# a. Create comparison DataFrame
comparison_df = pd.DataFrame({
    'original_text': df['review_text'],
    'processed_text': df['processed_text'],
    'sentiment': df['sentiment'],
    'num_sentences': df['tokenized_sentences'].apply(len),
    'num_words_before': df['tokenized_words'].apply(len),
    'num_words_after': df['tokenized_words_processed'].apply(len)
})

print("Comparison DataFrame created!")
print(f"Shape: {comparison_df.shape}")

# b. Calculate and display statistics
print("\n" + "="*80)
print("PREPROCESSING STATISTICS")
print("="*80)

avg_sentences = comparison_df['num_sentences'].mean()
avg_words_before = comparison_df['num_words_before'].mean()
avg_words_after = comparison_df['num_words_after'].mean()
reduction_percentage = ((avg_words_before - avg_words_after) / avg_words_before) * 100

print(f"\nAverage number of sentences per review: {avg_sentences:.2f}")
print(f"Average word count BEFORE preprocessing: {avg_words_before:.2f}")
print(f"Average word count AFTER preprocessing: {avg_words_after:.2f}")
print(f"Average reduction in word count: {reduction_percentage:.2f}%")

print(f"\nTotal reviews processed: {len(comparison_df):,}")
print(f"Total unique words in vocabulary: {len(vectorizer.get_feature_names_out())}")

# c. Save processed data
print("\n" + "="*80)
print("SAVING FILES...")
print("="*80)

# Save comparison data
comparison_df.to_csv('processed_reviews_comparison.csv', index=False)
print("✓ Saved: processed_reviews_comparison.csv")

# Save bag-of-words representation (word frequencies)
word_freq_df.to_csv('bow_representation.csv', index=False)
print("✓ Saved: bow_representation.csv")

# Also save the full bow matrix as CSV (sparse representation)
bow_df = pd.DataFrame(bow_matrix.toarray(), columns=vectorizer.get_feature_names_out())
bow_df.to_csv('bow_matrix_full.csv', index=False)
print("✓ Saved: bow_matrix_full.csv (full document-term matrix)")

print("\nAll files saved successfully!")

# d. Display sample comparisons
print("\n" + "="*80)
print("SAMPLE COMPARISONS (First 3 Reviews)")
print("="*80)

for idx in range(3):
    print(f"\n{'='*80}")
    print(f"REVIEW {idx + 1} - Sentiment: {comparison_df['sentiment'].iloc[idx].upper()}")
    print(f"{'='*80}")
    
    print(f"\nStatistics:")
    print(f"  - Sentences: {comparison_df['num_sentences'].iloc[idx]}")
    print(f"  - Words before: {comparison_df['num_words_before'].iloc[idx]}")
    print(f"  - Words after: {comparison_df['num_words_after'].iloc[idx]}")
    
    print(f"\nOriginal (first 250 chars):")
    print(comparison_df['original_text'].iloc[idx][:250] + "...")
    
    print(f"\nProcessed (first 250 chars):")
    print(comparison_df['processed_text'].iloc[idx][:250] + "...")

print("\n" + "="*80)
print("PROJECT COMPLETED SUCCESSFULLY!")
print("="*80)

Comparison DataFrame created!
Shape: (25000, 6)

PREPROCESSING STATISTICS

Average number of sentences per review: 10.84
Average word count BEFORE preprocessing: 282.61
Average word count AFTER preprocessing: 124.25
Average reduction in word count: 56.03%

Total reviews processed: 25,000
Total unique words in vocabulary: 1000

SAVING FILES...
✓ Saved: processed_reviews_comparison.csv
✓ Saved: bow_representation.csv
✓ Saved: bow_matrix_full.csv (full document-term matrix)

All files saved successfully!

SAMPLE COMPARISONS (First 3 Reviews)

REVIEW 1 - Sentiment: NEGATIVE

Statistics:
  - Sentences: 9
  - Words before: 335
  - Words after: 152

Original (first 250 chars):
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a ...

Processed (first 250 chars):
rented curiousyellow video store cont